In [ ]:
import pandas as pd
import json
from pathlib import Path
import plotly.express as px

In [ ]:


# Replace with the actual paths copied from VS Code
csv_path = Path("../data/planning_area_coverage_pressure.csv")

geojson_path = Path(
    "../data/MasterPlan2025PlanningAreaBoundaryNoSea.geojson"
)

# Load data
df = pd.read_csv(csv_path)

with open(geojson_path, "r", encoding="utf-8") as f:
    geojson = json.load(f)

# Standardise planning area names
df["planning_area"] = (
    df["planning_area"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Validate before mapping
assert len(df) == 32, "Expected 32 planning-area rows"
assert df["planning_area"].is_unique, "Duplicate planning areas found"

geojson_names = {
    feature["properties"]["PLN_AREA_N"]
    for feature in geojson["features"]
}

unmatched = set(df["planning_area"]) - geojson_names

assert not unmatched, f"Unmatched planning areas: {unmatched}"

print("Planning areas:", len(df))
print("URA geographic features:", len(geojson["features"]))
print("All planning area names matched successfully")

In [ ]:
# Folder for the final interactive maps
output_dir = csv_path.parent / "visualisations"
output_dir.mkdir(parents=True, exist_ok=True)

def create_singapore_map(column, title, legend_title, filename):

    fig = px.choropleth_map(
        df,
        geojson=geojson,
        locations="planning_area",
        featureidkey="properties.PLN_AREA_N",
        color=column,
        range_color=(0, 100),
        hover_name="planning_area",
        hover_data={
            "total_carparks": True,
            "carparks_with_strong_ev_candidates": True,
            "matched_charging_points": True,
            "candidate_coverage_pct": ":.2f",
            "occupancy_pct": ":.2f",
            "not_available_pct": ":.2f"
        },
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        center={"lat": 1.35, "lon": 103.82},
        zoom=10.2,
        opacity=0.75,
        labels={column: legend_title},
        title=title
    )

    fig.update_layout(
        height=700,
        margin=dict(l=0, r=0, t=60, b=0)
    )

    fig.show()

    output_file = output_dir / filename

    fig.write_html(
        str(output_file),
        include_plotlyjs=True
    )

    print("Saved:", output_file)


# Map 1 — Historical charging occupancy
create_singapore_map(
    column="occupancy_pct",
    title="EV Charging Occupancy by URA Planning Area",
    legend_title="Occupancy (%)",
    filename="Singapore_EV_Occupancy_Map.html"
)

# Map 2 — HDB candidate charging coverage
create_singapore_map(
    column="candidate_coverage_pct",
    title="HDB EV Charging Candidate Coverage by URA Planning Area",
    legend_title="Candidate coverage (%)",
    filename="Singapore_EV_Coverage_Map.html"
)